# RCFD vs CCFD — 3-way Comparison

Compares three model groups:

1. **Initial RCFD** (24 models, `model_utils_rcfd.py`) — lightweight early encoder + dual-backbone main branch.  
   Keys: `{enc}_rcfd_{bb}[_supcon_mtl|_supcon2_mtl|_supcon3_mtl]`

2. **New RCFD** (8 models, `03d_ccfd_training.py`) — dual-backbone early reg → FiLM → dual-backbone cls.  
   Keys: `rcfd_{bb}[_supcon_mtl|_supcon2_mtl|_supcon3_mtl]`

3. **New CCFD** (8 models, `03d_ccfd_training.py`) — dual-backbone early cls → FiLM → dual-backbone reg.  
   Keys: `ccfd_{bb}[_supcon_mtl|_supcon2_mtl|_supcon3_mtl]`

Absence of `{enc}_` prefix = dual-backbone early branch (new models).

In [ ]:
import os, sys
from pathlib import Path
try:
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        os.chdir(os.path.dirname(os.path.dirname(notebook_path)))
except Exception:
    pass
sys.path.insert(0, 'utils')
sys.path.insert(0, 'utils/model_training')
print('cwd:', os.getcwd())

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import importlib.util as _ilu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import accuracy_score

import config
from pipeline_utils import get_exp_paths
from model_utils_rcfd import ALL_RCFD_KEYS

# Import new model maps from 03d without triggering TF graph construction
_spec = _ilu.spec_from_file_location('_03d', Path(os.getcwd()) / '03d_ccfd_training.py')
_03d  = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_03d)

ALL_CCFD_KEYS   = _03d.ALL_CCFD_KEYS
ALL_RCFD2_KEYS  = _03d.ALL_RCFD2_KEYS
_CCFD_KEY_MAP   = _03d._CCFD_KEY_MAP
_RCFD2_KEY_MAP  = _03d._RCFD2_KEY_MAP
_CCFD_PRINT_MAP = _03d._CCFD_PRINT_MAP
_RCFD2_PRINT_MAP= _03d._RCFD2_PRINT_MAP
REG_SENTINEL    = _03d.REG_SENTINEL

# Initial RCFD maps come from config
_INIT_RCFD_KEY_MAP   = {k: v for k, v in config.MODEL_KEY_MAP.items()   if 'rcfd' in k}
_INIT_RCFD_PRINT_MAP = {k: v for k, v in config.MODEL_PRINT_MAP.items() if 'rcfd' in k}

ENC_TYPES = ['cnn', 'gru', 'trans']
BB_TYPES  = ['cgd', 'ctd']
SC_SUFFIX = ['', '_supcon_mtl', '_supcon2_mtl', '_supcon3_mtl']
SC_LABELS = ['SC0', 'SC1', 'SC2', 'SC3']

print(f'Init RCFD:  {len(ALL_RCFD_KEYS)} models')
print(f'New RCFD:   {len(ALL_RCFD2_KEYS)} models')
print(f'New CCFD:   {len(ALL_CCFD_KEYS)} models')

## Configuration

In [ ]:
# Set to None to aggregate across all experiments; or a specific exp name to inspect one
TARGET_EXP  = None
# N_SPLITS    = 5
N_SPLITS    = 1
CURVE_TYPE  = 'ori_curve'
FILTER      = None    # None = baseline (no outlier filter)

RESULT_JL = (config.TRAINING_10FOLD_RESULT_PATH if N_SPLITS > 1
             else config.TRAINING_RESULT_PATH)

exp_paths = get_exp_paths(config.DEFAULT_EXP_FOLDER)
if TARGET_EXP:
    exp_paths = [p for p in exp_paths if p.name == TARGET_EXP]

print(f'Evaluating {len(exp_paths)} experiment(s)')
for p in exp_paths:
    print(f'  {p.name}')

## Load results

In [ ]:
def _resolve_filter_entry(exp_path, result_jl, curve_type, filter_key):
    """Return the flat filter-level dict from a results joblib, or None."""
    fpath = exp_path / result_jl
    if not fpath.exists():
        return None
    try:
        all_res = joblib.load(fpath)
    except Exception:
        return None
    for title_res in all_res.values():
        for mode_res in title_res.values():
            if isinstance(mode_res, dict) and filter_key in mode_res:
                return mode_res[filter_key]
    return None


entries = {}
for ep in exp_paths:
    entry = _resolve_filter_entry(ep, RESULT_JL, CURVE_TYPE, FILTER)
    if entry and 'y_trues_' in entry:
        entries[ep.name] = entry
        print(f'  [OK] {ep.name}')
    else:
        print(f'  [--] {ep.name} (no matching entry)')

print(f'\nLoaded {len(entries)} entries')

## Helper functions

In [ ]:
def _mean_acc(y_trues, y_preds):
    if not y_trues or y_preds is None:
        return np.nan
    return float(np.mean([accuracy_score(yt, yp) for yt, yp in zip(y_trues, y_preds)]))


def _mean_rmse(reg_trues_folds, reg_preds_folds):
    if not reg_trues_folds or not reg_preds_folds:
        return np.nan
    rmses = []
    for rt, rp in zip(reg_trues_folds, reg_preds_folds):
        rt, rp = np.asarray(rt, dtype=float), np.asarray(rp, dtype=float)
        valid  = (rt != REG_SENTINEL) & np.isfinite(rt) & np.isfinite(rp)
        if valid.sum() < 2:
            continue
        rmses.append(np.sqrt(np.mean((rt[valid] - rp[valid]) ** 2)))
    return float(np.mean(rmses)) if rmses else np.nan


def _collect_all_metrics(entries, key_map, reg_prefix_fn=None):
    """Return dict[mkey] = dict(acc=[...], rmse=[...]) across experiments."""
    out = {mk: {'acc': [], 'rmse': []} for mk in key_map}
    for exp_name, entry in entries.items():
        y_trues = entry.get('y_trues_', [])
        for mk, (pk, probk, clsk) in key_map.items():
            out[mk]['acc'].append(_mean_acc(y_trues, entry.get(pk)))
            if reg_prefix_fn:
                rtk = reg_prefix_fn(mk)
                rpk = rtk.replace('y_reg_trues_', 'y_reg_preds_')
                out[mk]['rmse'].append(_mean_rmse(entry.get(rtk, []), entry.get(rpk, [])))
    return out


# Build key maps with explicit reg key names
def _init_reg_trues(mk): return f'y_reg_trues_{mk}_'
def _new_reg_trues(mk):  return f'y_reg_trues_{mk}_'

metrics_init  = _collect_all_metrics(entries, _INIT_RCFD_KEY_MAP,  _init_reg_trues)
metrics_new_r = _collect_all_metrics(entries, _RCFD2_KEY_MAP,       _new_reg_trues)
metrics_new_c = _collect_all_metrics(entries, _CCFD_KEY_MAP,        _new_reg_trues)


def _agg(d):
    """dict[mk] = dict(acc=...) → dict[mk] = (acc_mean, acc_std, rmse_mean, rmse_std, n)"""
    return {
        mk: (
            np.nanmean(v['acc']),  np.nanstd(v['acc']),
            np.nanmean(v['rmse']), np.nanstd(v['rmse']),
            int(np.sum(~np.isnan(v['acc']))),
        )
        for mk, v in d.items()
    }


agg_init  = _agg(metrics_init)
agg_new_r = _agg(metrics_new_r)
agg_new_c = _agg(metrics_new_c)

print('Aggregation done.')
init_available  = sum(1 for mk, (a,_,r,_,n) in agg_init.items()  if not np.isnan(a))
new_r_available = sum(1 for mk, (a,_,r,_,n) in agg_new_r.items() if not np.isnan(a))
new_c_available = sum(1 for mk, (a,_,r,_,n) in agg_new_c.items() if not np.isnan(a))
print(f'Init RCFD models with results:  {init_available}/{len(agg_init)}')
print(f'New RCFD models with results:   {new_r_available}/{len(agg_new_r)}')
print(f'New CCFD models with results:   {new_c_available}/{len(agg_new_c)}')

## Plot 1: Classification Accuracy — 3-way by backbone + SC level

- **Init RCFD**: best enc_type (cnn/gru/trans) for each backbone+SC
- **New RCFD**: dual-backbone early reg (primary cls task)
- **New CCFD**: dual-backbone early cls (auxiliary cls task, from early backbone A)

In [ ]:
def _best_init_rcfd_val(agg_dict, bb, sc_suf, col=0):
    """Best (max) value across enc_types for given bb + SC suffix."""
    vals = [agg_dict.get(f'{enc}_rcfd_{bb}{sc_suf}', (np.nan,)*5)[col]
            for enc in ENC_TYPES]
    finite = [v for v in vals if not np.isnan(v)]
    return max(finite) if finite else np.nan


fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=False)
fig.suptitle('Classification Accuracy — 3-way Comparison (mean across experiments)', fontsize=11)

bar_w = 0.25
group_colors = {'Init RCFD (best enc)': '#6baed6', 'New RCFD': '#2ca02c', 'New CCFD (aux)': '#d62728'}
x_pos = np.arange(len(BB_TYPES))

for si, (sc_lbl, sc_suf) in enumerate(zip(SC_LABELS, SC_SUFFIX)):
    ax = axes[si]
    init_vals  = [_best_init_rcfd_val(agg_init,  bb, sc_suf, col=0) * 100 for bb in BB_TYPES]
    new_r_vals = [agg_new_r.get(f'rcfd_{bb}{sc_suf}', (np.nan,)*5)[0] * 100 for bb in BB_TYPES]
    new_c_vals = [agg_new_c.get(f'ccfd_{bb}{sc_suf}', (np.nan,)*5)[0] * 100 for bb in BB_TYPES]

    for off, vals, lbl in zip([-bar_w, 0, bar_w],
                               [init_vals, new_r_vals, new_c_vals],
                               group_colors):
        ax.bar(x_pos + off, vals, width=bar_w, label=lbl,
               color=group_colors[lbl], alpha=0.85, edgecolor='white')

    ax.set_xticks(x_pos)
    ax.set_xticklabels([bb.upper() for bb in BB_TYPES])
    ax.set_title(sc_lbl, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 110)
    ax.set_ylabel('Acc (%)' if si == 0 else '')
    ax.grid(axis='y', alpha=0.3)
    if si == 0:
        ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('notebooks/rcfd_vs_ccfd_cls_acc_3way.pdf', bbox_inches='tight')
plt.show()

## Plot 2: Regression RMSE — 3-way by backbone + SC level

- **Init RCFD**: best enc_type (lowest RMSE) per backbone+SC — reg is auxiliary
- **New RCFD**: dual-backbone early reg — reg is auxiliary
- **New CCFD**: dual-backbone early cls — reg is PRIMARY

In [ ]:
def _best_init_rcfd_rmse(agg_dict, bb, sc_suf):
    """Best (min) RMSE across enc_types."""
    vals = [agg_dict.get(f'{enc}_rcfd_{bb}{sc_suf}', (np.nan,)*5)[2]
            for enc in ENC_TYPES]
    finite = [v for v in vals if not np.isnan(v)]
    return min(finite) if finite else np.nan


fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=False)
fig.suptitle('Regression RMSE — 3-way Comparison (mean across experiments, lower = better)', fontsize=11)

for si, (sc_lbl, sc_suf) in enumerate(zip(SC_LABELS, SC_SUFFIX)):
    ax = axes[si]
    init_vals  = [_best_init_rcfd_rmse(agg_init,  bb, sc_suf) for bb in BB_TYPES]
    new_r_vals = [agg_new_r.get(f'rcfd_{bb}{sc_suf}', (np.nan,)*5)[2] for bb in BB_TYPES]
    new_c_vals = [agg_new_c.get(f'ccfd_{bb}{sc_suf}', (np.nan,)*5)[2] for bb in BB_TYPES]

    for off, vals, lbl in zip([-bar_w, 0, bar_w],
                               [init_vals, new_r_vals, new_c_vals],
                               group_colors):
        ax.bar(x_pos + off, vals, width=bar_w, label=lbl,
               color=group_colors[lbl], alpha=0.85, edgecolor='white')

    ax.set_xticks(x_pos)
    ax.set_xticklabels([bb.upper() for bb in BB_TYPES])
    ax.set_title(sc_lbl, fontsize=10, fontweight='bold')
    ax.set_ylabel('RMSE' if si == 0 else '')
    ax.grid(axis='y', alpha=0.3)
    if si == 0:
        ax.legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig('notebooks/rcfd_vs_ccfd_reg_rmse_3way.pdf', bbox_inches='tight')
plt.show()

## Plot 3: New RCFD vs New CCFD — Symmetric Delta

Positive Δ cls = New RCFD better at classification (expected).  
Negative Δ RMSE = New CCFD better at regression (expected).

In [ ]:
delta_labels, delta_cls, delta_rmse = [], [], []

for bb in BB_TYPES:
    for sc_lbl, sc_suf in zip(SC_LABELS, SC_SUFFIX):
        rk = f'rcfd_{bb}{sc_suf}'
        ck = f'ccfd_{bb}{sc_suf}'
        r_acc  = agg_new_r.get(rk, (np.nan,)*5)[0]
        c_acc  = agg_new_c.get(ck, (np.nan,)*5)[0]
        r_rmse = agg_new_r.get(rk, (np.nan,)*5)[2]
        c_rmse = agg_new_c.get(ck, (np.nan,)*5)[2]
        delta_labels.append(f'{bb.upper()}\n{sc_lbl}')
        delta_cls.append((r_acc - c_acc) * 100 if not (np.isnan(r_acc) or np.isnan(c_acc)) else np.nan)
        delta_rmse.append(r_rmse - c_rmse if not (np.isnan(r_rmse) or np.isnan(c_rmse)) else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('New RCFD − New CCFD symmetric delta', fontsize=11)

x = np.arange(len(delta_labels))

for ax, deltas, ylabel, title in zip(
    axes,
    [delta_cls, delta_rmse],
    ['Δ Acc (pp)', 'Δ RMSE'],
    ['Classification: RCFD−CCFD  (positive → RCFD better)',
     'Regression RMSE: RCFD−CCFD  (positive → RCFD worse / CCFD better)'],
):
    colors = ['#2ca02c' if (v is not None and not np.isnan(v) and v > 0)
              else '#d62728' for v in deltas]
    ax.bar(x, deltas, color=colors, alpha=0.8, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels(delta_labels, fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/rcfd_vs_ccfd_delta.pdf', bbox_inches='tight')
plt.show()

## Plot 4: Init RCFD vs New RCFD — Does a dual-backbone early regressor help?

Both groups have classification as primary task. The only difference is early branch capacity.  
New RCFD uses a full dual backbone; init RCFD uses a lightweight encoder.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=False)
fig.suptitle('Init RCFD (best enc) vs New RCFD — Classification Accuracy', fontsize=11)

for si, (sc_lbl, sc_suf) in enumerate(zip(SC_LABELS, SC_SUFFIX)):
    ax = axes[si]
    for enc in ENC_TYPES:
        vals = [agg_init.get(f'{enc}_rcfd_{bb}{sc_suf}', (np.nan,)*5)[0] * 100 for bb in BB_TYPES]
        ax.plot([bb.upper() for bb in BB_TYPES], vals, 'o--', alpha=0.6, label=f'Init {enc.upper()}')
    new_r_vals = [agg_new_r.get(f'rcfd_{bb}{sc_suf}', (np.nan,)*5)[0] * 100 for bb in BB_TYPES]
    ax.plot([bb.upper() for bb in BB_TYPES], new_r_vals, 's-', color='#2ca02c',
            linewidth=2, markersize=8, label='New RCFD (dual-bb)', zorder=5)
    ax.set_title(sc_lbl, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 110)
    ax.set_ylabel('Acc (%)' if si == 0 else '')
    ax.grid(alpha=0.3)
    if si == 0:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('notebooks/init_vs_new_rcfd_cls.pdf', bbox_inches='tight')
plt.show()

## Plot 5: SC level benefit per group

How much does SC1/2/3 improve over SC0 for each group?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('SupCon Benefit (SC_n − SC_0) per group and backbone', fontsize=11)

sc_colors = {'SC1': '#fdae61', 'SC2': '#f46d43', 'SC3': '#d73027'}

def _mean_across_bb(agg, prefix, sc_suf, col):
    vals = [agg.get(f'{prefix}_{bb}{sc_suf}', (np.nan,)*5)[col] for bb in BB_TYPES]
    finite = [v for v in vals if not np.isnan(v)]
    return np.mean(finite) if finite else np.nan

def _mean_init(agg, sc_suf, col):
    vals = [agg.get(f'{enc}_rcfd_{bb}{sc_suf}', (np.nan,)*5)[col]
            for enc in ENC_TYPES for bb in BB_TYPES]
    finite = [v for v in vals if not np.isnan(v)]
    return np.mean(finite) if finite else np.nan

groups = [
    ('Init RCFD', None),
    ('New RCFD',  'rcfd'),
    ('New CCFD',  'ccfd'),
]

for col_i, (metric_label, metric_col) in enumerate([('Δ Acc (pp)', 0), ('Δ RMSE', 2)]):
    ax = axes[col_i]
    scale = 100 if metric_col == 0 else 1

    x = np.arange(len(groups))
    bar_w2 = 0.25

    for i, (sc_n, sc_suf) in enumerate(zip(['SC1', 'SC2', 'SC3'],
                                            ['_supcon_mtl', '_supcon2_mtl', '_supcon3_mtl'])):
        deltas = []
        for grp_name, prefix in groups:
            if prefix is None:
                v0  = _mean_init(agg_init, '', metric_col)
                vsc = _mean_init(agg_init, sc_suf, metric_col)
            else:
                agg_d = agg_new_r if prefix == 'rcfd' else agg_new_c
                v0    = _mean_across_bb(agg_d, prefix, '', metric_col)
                vsc   = _mean_across_bb(agg_d, prefix, sc_suf, metric_col)
            d = (vsc - v0) * scale if not (np.isnan(v0) or np.isnan(vsc)) else np.nan
            deltas.append(d)

        ax.bar(x + (i - 1) * bar_w2, deltas, width=bar_w2,
               label=sc_n, color=sc_colors[sc_n], alpha=0.85, edgecolor='white')

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels([g for g, _ in groups])
    ax.set_ylabel(metric_label)
    ax.set_title(f'{metric_label}: SC benefit over SC0 (averaged over bb)', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/rcfd_vs_ccfd_sc_benefit_3way.pdf', bbox_inches='tight')
plt.show()

## Summary Table

In [ ]:
rows = []
for mk, (am, as_, rm, rs, n) in agg_init.items():
    rows.append({'group': 'Init RCFD', 'model': mk, 'acc': am*100, 'acc_std': as_*100,
                 'rmse': rm, 'rmse_std': rs, 'n_exps': n})
for mk, (am, as_, rm, rs, n) in agg_new_r.items():
    rows.append({'group': 'New RCFD', 'model': mk, 'acc': am*100, 'acc_std': as_*100,
                 'rmse': rm, 'rmse_std': rs, 'n_exps': n})
for mk, (am, as_, rm, rs, n) in agg_new_c.items():
    rows.append({'group': 'New CCFD', 'model': mk, 'acc': am*100, 'acc_std': as_*100,
                 'rmse': rm, 'rmse_std': rs, 'n_exps': n})

df = (pd.DataFrame(rows)
      .dropna(subset=['acc'])
      .sort_values('acc', ascending=False)
      .reset_index(drop=True))
df.index += 1

print('=== Classification Accuracy Leaderboard (top 20) ===')
display(df[['group', 'model', 'acc', 'acc_std', 'rmse', 'n_exps']]
        .head(20)
        .round({'acc': 2, 'acc_std': 2, 'rmse': 4}))

In [ ]:
print('=== Regression RMSE Leaderboard — New CCFD only (primary task, lower = better) ===')
df_reg = (pd.DataFrame([
    {'model': mk, 'acc': am*100, 'rmse': rm, 'n_exps': n}
    for mk, (am, as_, rm, rs, n) in agg_new_c.items()
])
.dropna(subset=['rmse'])
.sort_values('rmse')
.reset_index(drop=True))
df_reg.index += 1
display(df_reg.round({'acc': 2, 'rmse': 4}))

In [ ]:
# Win summary: new RCFD vs new CCFD
rcfd_cls_wins = ccfd_cls_wins = rcfd_reg_wins = ccfd_reg_wins = 0
n_pairs = 0
for bb in BB_TYPES:
    for sc_suf in SC_SUFFIX:
        rk = f'rcfd_{bb}{sc_suf}'
        ck = f'ccfd_{bb}{sc_suf}'
        r_acc  = agg_new_r.get(rk, (np.nan,)*5)[0]
        c_acc  = agg_new_c.get(ck, (np.nan,)*5)[0]
        r_rmse = agg_new_r.get(rk, (np.nan,)*5)[2]
        c_rmse = agg_new_c.get(ck, (np.nan,)*5)[2]
        if not (np.isnan(r_acc) or np.isnan(c_acc)):
            n_pairs += 1
            if r_acc > c_acc: rcfd_cls_wins += 1
            else:              ccfd_cls_wins += 1
        if not (np.isnan(r_rmse) or np.isnan(c_rmse)):
            if c_rmse < r_rmse: ccfd_reg_wins += 1
            else:                rcfd_reg_wins += 1

print(f'=== New RCFD vs New CCFD win summary (n={n_pairs} pairs) ===')
print(f'Classification (primary for RCFD):  RCFD wins {rcfd_cls_wins}/{n_pairs}  |  CCFD wins {ccfd_cls_wins}/{n_pairs}')
print(f'Regression RMSE (primary for CCFD): CCFD wins {ccfd_reg_wins}/{n_pairs}  |  RCFD wins {rcfd_reg_wins}/{n_pairs}')